# Standardized PCA targets: dimension sweep for P0/P2/P4 and P0/P2

This notebook follows the first PCA-dimension comparison. It tests whether the poor neural-network results at 12–48 PCA components were caused by asking the MLP to predict PCA coordinates with very different scales.

**Core change:** after fitting PCA on training cosmologies, standardize each PCA coordinate using a `StandardScaler` fitted only on the training latent coordinates. The network predicts standardized coordinates; predictions are unscaled before inverse PCA.

Both target families use the same cosmology split, PCA dimensions, network architecture, optimizer settings, and three network random seeds.

- Targets: P0/P2/P4 and P0/P2
- PCA dimensions: 8, 12, 16, 24, 32, 48, 64, 80, 96
- Network seeds: 11, 22, 33 (initialization/shuffling only; the holdout split stays fixed)
- MLP: 128 x 128, tanh, Adam

For each run, the notebook records PCA-only table error, NN table error, reconstructed-spectrum error, per-multipole spectrum errors, per-component latent prediction errors, and training/validation loss histories.

The spectrum comparison uses the exact standard-basis nuisance contraction from the previous diagnostic notebook. It does **not** initialize CLASS or TheoryManager; it compares true and predicted held-out PT tables after applying the same fixed nuisance vector. Run from the project checkout in the configured NERSC environment.

In [ ]:
from pathlib import Path
import json
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.neural_network import MLPRegressor

np.set_printoptions(precision=6, suppress=True)

RANDOM_SPLIT_SEED = 20261112
NETWORK_SEEDS = [11, 22, 33]
PCA_DIMENSIONS = [8, 12, 16, 24, 32, 48, 64, 80, 96]
TEST_SIZE = 0.20

HIDDEN_LAYERS = (128, 128)
ACTIVATION = "tanh"
ALPHA = 1e-4
LEARNING_RATE = 3e-4
MAX_EPOCHS = 3000
PATIENCE = 150
MIN_IMPROVEMENT = 1e-6

# Partial_fit is called once per epoch so that we can record both training
# and validation latent-space MSE and implement transparent early stopping.
MLP_BASE_KWARGS = dict(
    hidden_layer_sizes=HIDDEN_LAYERS,
    activation=ACTIVATION,
    solver="adam",
    alpha=ALPHA,
    learning_rate_init=LEARNING_RATE,
    max_iter=1,
    shuffle=True,
    early_stopping=False,
    tol=0.0,
    n_iter_no_change=MAX_EPOCHS + 1,
    verbose=False,
)

## 1. Load and validate the completed training dataset

The source files contain all three multipoles with shape (3, 117, 19). We load the tables once, then derive P0/P2 by selecting the first two multipoles. Both emulators therefore use precisely the same cosmologies and source files.

In [ ]:
DATA_ROOT = Path("./training_data")
RUN_NAME = "velocileptors_pt_lhs_v2_restricted"
RUN_DIR = DATA_ROOT / RUN_NAME
TABLE_DIR = RUN_DIR / "pt_tables"

if not TABLE_DIR.is_dir():
    raise FileNotFoundError(f"PT-table directory not found: {TABLE_DIR.resolve()}")

with open(RUN_DIR / "run_config.json") as stream:
    run_config = json.load(stream)

COSMOLOGY_PARAMETERS = [
    "omega_b", "omega_cdm", "sigma8_cb", "n_s",
    "alpha_s", "N_ur", "w0_fld", "wa_fld",
]
SOURCE_ELL = np.array([0, 2, 4], dtype=int)
SOURCE_SHAPE = (3, 117, 19)

table_files = sorted(TABLE_DIR.glob("cosmo_*.npz"))
if not table_files:
    raise FileNotFoundError(f"No cosmo_*.npz files found in {TABLE_DIR}")

records, tables = [], []
native_k_reference = None
component_labels_reference = None

for path in table_files:
    with np.load(path, allow_pickle=False) as saved:
        required = {"pt_table", "k_native", "ells", "component_labels",
                    "cosmology_json", "metadata_json"}
        missing = required.difference(saved.files)
        if missing:
            raise KeyError(f"{path.name} is missing keys: {sorted(missing)}")
        table = np.asarray(saved["pt_table"], dtype=np.float64)
        k_native = np.asarray(saved["k_native"], dtype=np.float64)
        ells = np.asarray(saved["ells"], dtype=int)
        labels = saved["component_labels"].astype(str).tolist()
        cosmo = json.loads(str(saved["cosmology_json"].item()))
        metadata = json.loads(str(saved["metadata_json"].item()))

    if table.shape != SOURCE_SHAPE:
        raise ValueError(f"{path.name}: table shape {table.shape}, expected {SOURCE_SHAPE}")
    if not np.isfinite(table).all():
        raise ValueError(f"{path.name}: non-finite PT-table values")
    if k_native.shape != (SOURCE_SHAPE[1],) or not np.isfinite(k_native).all():
        raise ValueError(f"{path.name}: invalid native k coordinate")
    if not np.all(np.diff(k_native) > 0):
        raise ValueError(f"{path.name}: native k must be strictly increasing")
    if not np.array_equal(ells, SOURCE_ELL):
        raise ValueError(f"{path.name}: unexpected multipoles {ells}")
    if len(labels) != SOURCE_SHAPE[2]:
        raise ValueError(f"{path.name}: expected 19 component labels")

    if native_k_reference is None:
        native_k_reference = k_native.copy()
        component_labels_reference = labels
    else:
        if not np.allclose(k_native, native_k_reference, rtol=0, atol=1e-12):
            raise ValueError(f"{path.name}: native k grid differs")
        if labels != component_labels_reference:
            raise ValueError(f"{path.name}: component-label ordering changed")

    if not all(name in cosmo for name in COSMOLOGY_PARAMETERS):
        raise ValueError(f"{path.name}: missing cosmological coordinates")
    xrow = [float(cosmo[name]) for name in COSMOLOGY_PARAMETERS]
    if not np.isfinite(xrow).all():
        raise ValueError(f"{path.name}: non-finite cosmological coordinates")

    file_index = int(path.stem.split("_")[-1])
    if int(metadata.get("cosmology_index", -1)) != file_index:
        raise ValueError(f"{path.name}: filename and metadata indices disagree")

    records.append({"cosmology_index": file_index,
                    **dict(zip(COSMOLOGY_PARAMETERS, xrow))})
    tables.append(table)

order = np.argsort([r["cosmology_index"] for r in records])
X_all = pd.DataFrame(records).iloc[order].reset_index(drop=True)
Y_024 = np.stack(tables, axis=0)[order]

if X_all["cosmology_index"].duplicated().any():
    raise ValueError("Duplicate cosmology indices found")

X = X_all[COSMOLOGY_PARAMETERS].to_numpy(dtype=np.float64)
Y_02 = Y_024[:, :2, :, :].copy()
n_cosmo = len(X)

print("Run:", RUN_NAME)
print("Configured cosmologies:", run_config.get("n_cosmologies"))
print("Loaded cosmologies:", n_cosmo)
print("Input shape:", X.shape)
print("P0/P2/P4 table shape:", Y_024.shape)
print("P0/P2 table shape:", Y_02.shape)
print("Native k range:", (native_k_reference.min(), native_k_reference.max()))

## 2. Make one fixed train/holdout split

The three network seeds vary initialization and minibatch shuffling, **not** the cosmology split. Every target family, PCA dimension, and network seed is evaluated on the same held-out cosmologies. Input scaling is fitted on training cosmologies only.

In [ ]:
all_indices = np.arange(n_cosmo)
train_idx, valid_idx = train_test_split(
    all_indices, test_size=TEST_SIZE, random_state=RANDOM_SPLIT_SEED
)

X_train_raw, X_valid_raw = X[train_idx], X[valid_idx]
x_scaler = StandardScaler().fit(X_train_raw)
X_train = x_scaler.transform(X_train_raw)
X_valid = x_scaler.transform(X_valid_raw)

print("Training cosmologies:", len(train_idx))
print("Holdout cosmologies:", len(valid_idx))
print("Network seeds:", NETWORK_SEEDS)
print("PCA dimensions:", PCA_DIMENSIONS)

## 3. Define metrics and the standard-basis spectrum contraction

Component scaling and PCA are fitted using training cosmologies only. The contraction below deliberately matches the standard-basis nuisance evaluation used in the preceding comparison notebook. We apply one fixed standard nuisance vector to both the true and predicted tables; this isolates errors introduced by the emulator.

In [ ]:
from desilike.theories.galaxy_clustering.full_shape import (
    _velocileptors_combine_bias_terms_spectrum2_poles,
)
from src.theory import K_FID

FIDUCIAL_NUISANCE = {
    "b1": 2.0, "b2": 0.0, "bs": 0.0, "b3": 0.0,
    "alpha0": 0.0, "alpha2": 0.0, "alpha4": 0.0, "alpha6": 0.0,
    "sn0": 0.0, "sn2": 0.0, "sn4": 0.0,
}
STANDARD_PARAMETER_ORDER = [
    "b1", "b2", "bs", "b3",
    "alpha0", "alpha2", "alpha4", "alpha6",
    "sn0", "sn2", "sn4",
]
standard_pars = np.asarray(
    [FIDUCIAL_NUISANCE[name] for name in STANDARD_PARAMETER_ORDER],
    dtype=float,
)
K_FID = np.asarray(K_FID, dtype=float)

def error_summary(truth, prediction):
    diff = np.asarray(prediction) - np.asarray(truth)
    denom = max(float(np.sqrt(np.mean(np.asarray(truth)**2))), 1e-30)
    return {
        "relative_rms": float(np.sqrt(np.mean(diff**2)) / denom),
        "rmse": float(np.sqrt(np.mean(diff**2))),
        "max_abs": float(np.max(np.abs(diff))),
    }

def spectra_from_tables(table_batch):
    """Contract native PT tables with the fixed standard-basis nuisance vector."""
    spectra = []
    for table in table_batch:
        native_pk = np.asarray(
            _velocileptors_combine_bias_terms_spectrum2_poles(
                np.asarray(table, dtype=float), standard_pars
            ),
            dtype=float,
        )
        expected = (table.shape[0], len(native_k_reference))
        if native_pk.shape != expected:
            raise ValueError(
                f"Unexpected contraction shape {native_pk.shape}; expected {expected}"
            )
        output_pk = np.vstack([
            np.interp(K_FID, native_k_reference, row) for row in native_pk
        ])
        spectra.append(output_pk)
    return np.asarray(spectra)

print("Standard nuisance vector:", standard_pars)
print("Output k samples:", len(K_FID))

## 4. Fit PCA and standardize its coordinates

The important change from the previous notebook is the latent-coordinate scaler:

1. Fit PCA on the training PT tables.
2. Transform training and holdout tables into PCA coordinates.
3. Fit `StandardScaler` on **training PCA coordinates only**.
4. Train the MLP to predict standardized coordinates.
5. Undo latent scaling before applying the inverse PCA transform.

PCA scores have variances equal to their eigenvalues, so later coordinates can have much smaller numerical scales. Standardizing them gives each coordinate comparable scale in the MLP loss. The inverse scaling restores the original PCA coordinates exactly before table reconstruction.

In [ ]:
TARGETS = {
    "P0/P2/P4": (Y_024, [0, 2, 4]),
    "P0/P2": (Y_02, [0, 2]),
}

def prepare_target_arrays(Y_all):
    Y_train_table = Y_all[train_idx]
    Y_valid_table = Y_all[valid_idx]

    # Same component-wise RMS scaling used in the previous comparison.
    component_scale = np.sqrt(
        np.mean(Y_train_table**2, axis=(0, 1, 2))
    )
    component_scale = np.maximum(component_scale, 1e-30)

    def flatten_scaled(table_array):
        return (
            table_array / component_scale[None, None, None, :]
        ).reshape(len(table_array), -1)

    return (
        Y_train_table, Y_valid_table, component_scale,
        flatten_scaled(Y_train_table), flatten_scaled(Y_valid_table),
    )

## 5. Training loop: three seeds, nine dimensions, two target families

We use `partial_fit` one epoch at a time. This lets us calculate training and validation MSE on the standardized PCA coordinates after every epoch, implement early stopping on the fixed holdout set, and restore the best validation checkpoint. The holdout set is used for model selection here, so treat these results as a development diagnostic rather than an untouched final test.

The per-component diagnostic is the validation RMSE of each standardized PCA coordinate. A value near 1 means the prediction error for that coordinate is roughly one training-set standard deviation.

In [ ]:
def train_one_network(X_train, Z_train_scaled, X_valid, Z_valid_scaled, seed):
    kwargs = dict(MLP_BASE_KWARGS)
    kwargs["random_state"] = int(seed)
    model = MLPRegressor(**kwargs)

    train_history = []
    valid_history = []
    best_valid = np.inf
    best_epoch = 0
    best_weights = None
    epochs_without_improvement = 0

    for epoch in range(1, MAX_EPOCHS + 1):
        model.partial_fit(X_train, Z_train_scaled)

        train_pred = model.predict(X_train)
        valid_pred = model.predict(X_valid)
        train_mse = float(np.mean((train_pred - Z_train_scaled)**2))
        valid_mse = float(np.mean((valid_pred - Z_valid_scaled)**2))
        train_history.append(train_mse)
        valid_history.append(valid_mse)

        if valid_mse < best_valid - MIN_IMPROVEMENT:
            best_valid = valid_mse
            best_epoch = epoch
            best_weights = (
                [w.copy() for w in model.coefs_],
                [b.copy() for b in model.intercepts_],
            )
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= PATIENCE:
            break

    if best_weights is None:
        raise RuntimeError("Training did not produce a valid checkpoint.")

    # Restore the best validation checkpoint, not the final epoch.
    model.coefs_ = best_weights[0]
    model.intercepts_ = best_weights[1]

    history = pd.DataFrame({
        "epoch": np.arange(1, len(train_history) + 1),
        "train_mse": train_history,
        "valid_mse": valid_history,
    })
    return model, history, best_epoch

results = []
per_component_rows = []
per_ell_rows = []
histories = {}
fitted_models = {}

for target_name, (Y_all, multipoles) in TARGETS.items():
    print("\\n" + "=" * 76)
    print(f"Target: {target_name}; table shape: {Y_all.shape[1:]}")
    print("=" * 76)

    (Y_train_table, Y_valid_table, component_scale,
     Y_train_flat, Y_valid_flat) = prepare_target_arrays(Y_all)

    max_allowed = min(len(train_idx) - 1, Y_train_flat.shape[1])
    for n_components in PCA_DIMENSIONS:
        if n_components > max_allowed:
            print(f"Skipping {n_components}: maximum allowed is {max_allowed}")
            continue

        print(f"\\n{target_name}: {n_components} PCA components")
        pca = PCA(n_components=n_components, svd_solver="full")
        Z_train = pca.fit_transform(Y_train_flat)
        Z_valid = pca.transform(Y_valid_flat)

        # PCA-only reconstruction, prior to any NN prediction.
        Y_valid_pca_flat = pca.inverse_transform(Z_valid)
        Y_valid_pca = (
            Y_valid_pca_flat.reshape(Y_valid_table.shape)
            * component_scale[None, None, None, :]
        )
        pca_error = error_summary(Y_valid_table, Y_valid_pca)

        # Critical change: standardize latent coordinates, fit on train only.
        z_scaler = StandardScaler().fit(Z_train)
        Z_train_scaled = z_scaler.transform(Z_train)
        Z_valid_scaled = z_scaler.transform(Z_valid)

        for seed in NETWORK_SEEDS:
            start = time.perf_counter()
            mlp, history, best_epoch = train_one_network(
                X_train, Z_train_scaled, X_valid, Z_valid_scaled, seed
            )
            fit_seconds = time.perf_counter() - start

            Z_valid_pred_scaled = mlp.predict(X_valid)
            Z_valid_pred = z_scaler.inverse_transform(Z_valid_pred_scaled)
            Y_valid_pred_flat = pca.inverse_transform(Z_valid_pred)
            Y_valid_pred = (
                Y_valid_pred_flat.reshape(Y_valid_table.shape)
                * component_scale[None, None, None, :]
            )

            table_error = error_summary(Y_valid_table, Y_valid_pred)
            pk_true = spectra_from_tables(Y_valid_table)
            pk_pred = spectra_from_tables(Y_valid_pred)
            spectrum_error = error_summary(pk_true, pk_pred)

            run_key = (target_name, n_components, seed)
            histories[run_key] = history
            fitted_models[run_key] = {
                "pca": pca,
                "z_scaler": z_scaler,
                "mlp": mlp,
                "component_scale": component_scale.copy(),
                "Y_valid_pred": Y_valid_pred,
                "pk_true": pk_true,
                "pk_pred": pk_pred,
            }

            results.append({
                "target": target_name,
                "n_components": n_components,
                "seed": seed,
                "pca_variance_explained": float(
                    np.sum(pca.explained_variance_ratio_)
                ),
                "pca_table_relative_rms": pca_error["relative_rms"],
                "nn_table_relative_rms": table_error["relative_rms"],
                "nn_spectrum_relative_rms": spectrum_error["relative_rms"],
                "best_epoch": best_epoch,
                "epochs_run": len(history),
                "fit_seconds": fit_seconds,
            })

            # Per-PC errors in standardized latent units.
            component_rmse = np.sqrt(np.mean(
                (Z_valid_pred_scaled - Z_valid_scaled)**2, axis=0
            ))
            for pc_index, pc_rmse in enumerate(component_rmse):
                per_component_rows.append({
                    "target": target_name,
                    "n_components": n_components,
                    "seed": seed,
                    "pc_index": pc_index + 1,
                    "standardized_pc_rmse": float(pc_rmse),
                })

            # Per-multipole spectrum errors.
            for i, ell in enumerate(multipoles):
                ell_error = error_summary(pk_true[:, i, :], pk_pred[:, i, :])
                per_ell_rows.append({
                    "target": target_name,
                    "n_components": n_components,
                    "seed": seed,
                    "ell": ell,
                    "spectrum_relative_rms": ell_error["relative_rms"],
                })

            print(
                f"  seed={seed:2d} | best epoch={best_epoch:4d} | "
                f"PCA={pca_error['relative_rms']:.4e} | "
                f"table={table_error['relative_rms']:.4e} | "
                f"spectrum={spectrum_error['relative_rms']:.4e} | "
                f"{fit_seconds:.1f}s"
            )

results_df = pd.DataFrame(results)
per_component_df = pd.DataFrame(per_component_rows)
per_ell_df = pd.DataFrame(per_ell_rows)

summary_df = (
    results_df.groupby(["target", "n_components"])
    .agg(
        pca_table_relative_rms=("pca_table_relative_rms", "mean"),
        nn_table_mean=("nn_table_relative_rms", "mean"),
        nn_table_std=("nn_table_relative_rms", "std"),
        nn_spectrum_mean=("nn_spectrum_relative_rms", "mean"),
        nn_spectrum_std=("nn_spectrum_relative_rms", "std"),
        variance_explained=("pca_variance_explained", "mean"),
        best_epoch_mean=("best_epoch", "mean"),
        fit_seconds_mean=("fit_seconds", "mean"),
    )
    .reset_index()
)

display(summary_df)

## 6. Compare PCA-only and NN table errors

The PCA-only curve is independent of network initialization and therefore appears once per target/dimension. NN results show the mean across the three network seeds, with one-standard-deviation error bars. If target scaling fixes the optimization issue, the NN curves should track much closer to the PCA-only curves.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.3))

for target_name, group in summary_df.groupby("target"):
    group = group.sort_values("n_components")
    axes[0].plot(
        group["n_components"], group["pca_table_relative_rms"],
        marker="o", label=target_name,
    )
    axes[1].errorbar(
        group["n_components"], group["nn_table_mean"],
        yerr=group["nn_table_std"], marker="o", capsize=3,
        label=target_name,
    )

axes[0].set_title("PCA-only holdout reconstruction")
axes[0].set_ylabel("Relative RMS PT-table error")
axes[1].set_title("PCA + NN holdout prediction (3 seeds)")
axes[1].set_ylabel("Relative RMS PT-table error")
for ax in axes:
    ax.set_xlabel("Number of PCA components")
    ax.set_xticks(PCA_DIMENSIONS)
    ax.grid(alpha=0.25)
    ax.legend()
fig.tight_layout()
plt.show()

## 7. Reconstructed-spectrum errors

This is the principal practical metric. Error bars show the standard deviation across the three network seeds. The per-multipole plot makes it possible to see whether one multipole is driving the combined error.

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.5))
for target_name, group in summary_df.groupby("target"):
    group = group.sort_values("n_components")
    ax.errorbar(
        group["n_components"], group["nn_spectrum_mean"],
        yerr=group["nn_spectrum_std"], marker="o", capsize=3,
        label=target_name,
    )
ax.set_xlabel("Number of PCA components")
ax.set_ylabel("Relative RMS reconstructed-spectrum error")
ax.set_title("Holdout spectrum error (mean ± 1 std over seeds)")
ax.set_xticks(PCA_DIMENSIONS)
ax.grid(alpha=0.25)
ax.legend()
plt.show()

per_ell_summary = (
    per_ell_df.groupby(["target", "n_components", "ell"])
    ["spectrum_relative_rms"].agg(["mean", "std"]).reset_index()
)
fig, axes = plt.subplots(1, 3, figsize=(14, 3.9), sharey=True)
for ax, ell in zip(axes, [0, 2, 4]):
    subset = per_ell_summary[per_ell_summary["ell"] == ell]
    for target_name, group in subset.groupby("target"):
        group = group.sort_values("n_components")
        ax.errorbar(
            group["n_components"], group["mean"], yerr=group["std"],
            marker="o", capsize=2, label=target_name,
        )
    ax.set_title(f"ell={ell}")
    ax.set_xlabel("PCA components")
    ax.set_xticks(PCA_DIMENSIONS)
    ax.grid(alpha=0.25)
axes[0].set_ylabel("Relative RMS spectrum error")
axes[-1].legend()
fig.tight_layout()
plt.show()

## 8. Training versus validation loss

Each run records MSE in standardized PCA-coordinate space at every epoch. The plots below show the median training and validation histories across the three seeds for selected dimensions. A widening train/validation gap suggests overfitting; both losses remaining high suggests underfitting or an optimization problem.

The fixed holdout set is used for early stopping, so the validation curve is a model-selection diagnostic rather than an independent final-test estimate.

In [ ]:
DIAGNOSTIC_DIMS = [8, 24, 48, 96]

fig, axes = plt.subplots(
    len(TARGETS), len(DIAGNOSTIC_DIMS),
    figsize=(16, 7), squeeze=False, sharex=False, sharey=False,
)
for row, target_name in enumerate(TARGETS):
    for col, n_components in enumerate(DIAGNOSTIC_DIMS):
        ax = axes[row, col]
        run_histories = [
            histories[(target_name, n_components, seed)]
            for seed in NETWORK_SEEDS
            if (target_name, n_components, seed) in histories
        ]
        if not run_histories:
            ax.set_visible(False)
            continue
        min_len = min(len(h) for h in run_histories)
        train_curves = np.stack([
            h["train_mse"].to_numpy()[:min_len] for h in run_histories
        ])
        valid_curves = np.stack([
            h["valid_mse"].to_numpy()[:min_len] for h in run_histories
        ])
        epochs = np.arange(1, min_len + 1)
        ax.plot(epochs, np.median(train_curves, axis=0), label="train")
        ax.plot(epochs, np.median(valid_curves, axis=0), label="validation")
        ax.set_title(f"{target_name}, {n_components} PCs")
        ax.set_xlabel("Epoch")
        ax.set_ylabel("Standardized latent MSE")
        ax.grid(alpha=0.25)
        if row == 0 and col == 0:
            ax.legend()
fig.tight_layout()
plt.show()

## 9. Per-component prediction accuracy

This diagnostic answers whether the network predicts some PCA coordinates much worse than others. Errors are reported in units of each coordinate's training-set standard deviation (because the latent targets were standardized).

The heatmaps show the mean standardized-coordinate RMSE over the three network seeds. Large errors concentrated in later PCs would indicate that the network has particular difficulty learning low-variance directions; uniformly large errors would point to a broader mapping/optimization issue.

In [ ]:
pc_summary = (
    per_component_df.groupby(["target", "n_components", "pc_index"])
    ["standardized_pc_rmse"].mean().reset_index()
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, target_name in zip(axes, ["P0/P2/P4", "P0/P2"]):
    subset = pc_summary[pc_summary["target"] == target_name]
    matrix = subset.pivot(
        index="n_components", columns="pc_index",
        values="standardized_pc_rmse",
    ).sort_index()
    im = ax.imshow(matrix.to_numpy(), aspect="auto", origin="lower")
    ax.set_title(target_name)
    ax.set_xlabel("PCA component index")
    ax.set_ylabel("Number of retained components")
    ax.set_xticks(np.arange(matrix.shape[1]))
    ax.set_xticklabels(matrix.columns, rotation=90)
    ax.set_yticks(np.arange(matrix.shape[0]))
    ax.set_yticklabels(matrix.index)
    fig.colorbar(im, ax=ax, label="Standardized PC RMSE")
fig.suptitle("Per-component holdout prediction error (mean over seeds)")
fig.tight_layout()
plt.show()

# Also list the worst-predicted coordinates for each target/dimension.
worst_pc = (
    pc_summary.sort_values("standardized_pc_rmse", ascending=False)
    .groupby(["target", "n_components"]).head(5)
    .sort_values(["target", "n_components", "standardized_pc_rmse"],
                 ascending=[True, True, False])
)
display(worst_pc)

## 10. PCA variance and seed-to-seed variability

Variance explained remains a useful compression diagnostic, but not the selection criterion. The table and plots should be interpreted together: PCA-only reconstruction, NN prediction, final spectra, optimization histories, and variability across seeds.

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.2))
for target_name, group in summary_df.groupby("target"):
    group = group.sort_values("n_components")
    ax.plot(
        group["n_components"], group["variance_explained"],
        marker="o", label=target_name,
    )
ax.set_xlabel("Number of PCA components")
ax.set_ylabel("Cumulative variance explained")
ax.set_title("PCA variance retained")
ax.set_xticks(PCA_DIMENSIONS)
ax.set_ylim(0, 1.01)
ax.grid(alpha=0.25)
ax.legend()
plt.show()

display(results_df.sort_values(["target", "n_components", "seed"]))

## 11. Save the complete results

Four CSV files are written in the current working directory:
- run-level results (one row per target/dimension/seed)
- mean/std summary across seeds
- per-component latent-coordinate errors
- per-multipole spectrum errors

These can be compared directly with the previous single-seed sweep.

In [ ]:
results_df.to_csv("standardized_pca_sweep_runs.csv", index=False)
summary_df.to_csv("standardized_pca_sweep_summary.csv", index=False)
per_component_df.to_csv("standardized_pca_per_component_errors.csv", index=False)
per_ell_df.to_csv("standardized_pca_per_ell_errors.csv", index=False)

print("Saved result files in:", Path.cwd())
display(summary_df.pivot(
    index="n_components",
    columns="target",
    values=[
        "pca_table_relative_rms",
        "nn_table_mean",
        "nn_table_std",
        "nn_spectrum_mean",
        "nn_spectrum_std",
        "variance_explained",
    ],
))

## Interpretation guide

- **If NN error now follows PCA-only error:** target-scale imbalance was likely a major cause of the previous failure.
- **If training error is low but validation error is high:** the mapping is overfitting; consider regularization, a smaller network, or more training cosmologies.
- **If both training and validation errors remain high:** the current network/optimizer may be underfitting or training may be unstable.
- **If later PCs have much larger standardized errors:** the network has difficulty learning those directions even after scale normalization.
- **If seed-to-seed scatter is large:** the apparent dimension dependence may be optimization variability; don't select a dimension from one seed.
- **If PCA-only error keeps falling but spectrum error plateaus:** additional coefficient-table accuracy may have diminishing practical value for this fixed nuisance contraction.

A useful PCA dimension is the smallest one that achieves stable, sufficiently low reconstructed-spectrum error across seeds—not necessarily the dimension with the largest variance explained.

**Important:** this notebook is an experiment/diagnostic. It does not refit and export a production emulator on all cosmologies. Once a dimension and training recipe are selected, we should update the main training notebook accordingly.